In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from pydap.net import create_session
import json
import cartopy.crs as ccrs
import xarray as xr
import datetime as dt
from pydap.client import open_url, consolidate_metadata, get_cmr_urls
import earthaccess
import pydap

In [ ]:
print("pydap version: ", pydap.__version__)

<span style='color:#ff6666'><font size="5">**Finding Cloud OPeNDAP URLs with NASA's CMR**:

<span style='font-family:serif'> <font size="3"><span style='color:Black'> Below we illustrate how to find OPeNDAP URLs via the **CMR**

<span style='color:#0066cc'><font size="3.5"> **To find (cloud) OPeNDAP URL you will need:**

* One of `Collection Concept ID` or `dataset DOI`
* Time Range


Here, we will use the Collection Concept ID associated with the [Temperature and Salinity](https://podaac.jpl.nasa.gov/dataset/ECCO_L4_TEMP_SALINITY_LLC0090GRID_MONTHLY_V4R4). For example:

<img src="img/ECCO_conceptID_doi.png" alt="drawing" width="750"/>    





In [ ]:
session = requests.Session()

In [ ]:
# CMR API base url
cmrurl='https://cmr.earthdata.nasa.gov/search/'
doi = '10.5067/ECL5M-OTS44'

<span style='font-family:serif'> <font size="5.5"><span style='color:#0066cc'> **Specify time range**

<font size="3"><span style='color:Black'> This dataset covers `01-01-1992` to `01-18-2018`. 


In [ ]:
start_date =  dt.datetime(1992, 1, 1)
end_date = dt.datetime(2017, 12, 31)

time_range=[start_date,end_date] # One month of data


<span style='font-family:serif'> <font size="5.5"><span style='color:#0066cc'> **Get all available cloud OPeNDAP URLs via CMR**

The cell below will search/find all OPeNDAP URLs associated with the Collection concept ID.

The results wll be stored in the variable `granules_urls`.
    

In [ ]:
%%time
granules_urls = get_cmr_urls(doi=doi, time_range=time_range, limit=100)

In [ ]:
print("WE found: ", len(granules_urls), " total Cloud OPeNDAP URLS associated with this collection!")

<span style='font-family:serif'> <font size="5.5"><span style='color:#0066cc'> **Pydap Approach**

<span style='font-family:serif'> <font size="3.5"> We can use <span style='color:#ff6666'>**PyDAP**<span style='color:black'> to inspect the metadata associated with each of the urls.

<span style='font-family:serif'> <font size="3.5">Below we illustrate the use of <span style='color:#ff6666'>**PyDAP**<span style='color:black'> with Token authentication to access OPeNDAP metadata.

<span style='font-family:serif'> <font size="3.5"> This will be useful when accessing OPeNDAP URLs via xarray.


<span style='font-family:serif'> <font size="5.5"><span style='color:#0066cc'> **Import Token Authorization and create Session**

<span style='font-family:serif'> <font size="3.5"> Here, we use `earthaccess` to authenticate and extract the user token from EDL. It will all happen in the background, but you will be prompted to add your EDL credentials.


In [ ]:
auth = earthaccess.login(strategy="interactive", persist=True) # you will be promted to add your EDL credentials

# pass Token Authorization to a new Session.
cache_kwargs={'cache_name':'ECCOv4'}
my_session = create_session(use_cache=True, session=auth.get_session(), cache_kwargs=cache_kwargs)
my_session.cache.clear()

<span style='font-family:serif'> <font size="5.5"><span style='color:#0066cc'> **Lazy access to remote data via pydap's client API**

<font size="3"> <span style='color:#ff6666'>**PyDAP**<span style='color:black'> exploits the OPeNDAP's separation between metadata and data, to create lazy dataset objects that point to the data. These lazy objects contain all the attributes detailed in OPeNDAP's metadata files (DMR)

In [ ]:
%%time
pyds = open_url(granules_urls[0], session=my_session, protocol='dap4')

In [ ]:
pyds.tree()

<span style='font-family:serif'> <font size="5.5"><span style='color:#0066cc'> **Not all Variables are of interest. Lets use Constraint Expressions!**

<font size="3">  Consider that we only want
- `THETA`
- `SALT`

<font size="3">  and their `dimensions`. 

In [ ]:
print("dimension of THETA:" , pyds['THETA'].dims)
print("dimension of SALT:" , pyds['SALT'].dims)

<span style='color:#0066cc'><font size="5"> **Construct Constraint Expression**

<font size="3"> That will instruct the Hyrax Data Server to only give use our desired variables.

<font size="3">  This variable will be named `CE`. We will add it to each (granule) cloud OPeNDAP URL. THis will allow us to construct a `Data Cube`


In [ ]:
dims = pyds['SALT'].dims
Vars = ['/THETA', '/SALT'] + dims

# Below construct Contraint Expression
CE = "?dap4.ce="+(";").join(Vars)
print("constraint expression: ", CE)

In [ ]:
print(" Each Cloud OPeNDAP URL will look like: \n", granules_urls[0]+CE)

<span style='color:#0066cc'><font size="5"> **Construct DAP4 URLS:**
 

<font size="3"> A DAP4 url begins with `dap4` as a scheme. 

<font size="3"> **NOTE**: This is only for xarray and <span style='color:#ff6666'>**PyDAP**<span style='color:black'>.


In [ ]:
new_urls = [url.replace("https", "dap4") + CE for url in granules_urls][:100] # consider only the first 100 urls
new_urls[:4]

<span style='color:#0066cc'><font size="5"> **Consolidate all URL Metadata Associated with the Data URL of cloud OPeNDAP URLs**

<font size="3"> You can construct a persistent reference to all Cloud OPeNDAP urls for later use!!!! 


In [ ]:
# clear just in case
my_session.cache.clear()

In [ ]:
%%time
consolidate_metadata(new_urls, my_session, concat_dim='time')

## What happened?

<font size="3"> All necessary metadata was fetch from opendap servers, and it can be used and reused by xarray. 


## Create a datacube with xarray and pydap as an engine!


In [ ]:
%%time
ds = xr.open_mfdataset(new_urls, engine='pydap', session=my_session, parallel=True, combine='nested', concat_dim='time', chunks={'tile':1, 'k':1})
ds

## Download some data

So far, only metadata has been downloaded. Below we plot some data in the NorthAtlantic ocean





In [ ]:
%%time
ds['THETA'].isel(time=0, k=0, tile=2).plot(cmap='RdBu_r', vmin=-4, vmax=30);

In [ ]:
ds['THETA'].isel(time=0, k=0, tile=2).attrs